# Mốc 2 — kiểm runner đối chứng trên Indian Pines

**Một notebook mới, tự chứa. Không chạy lại notebook B hay smoke mốc 1.** Đây là kiểm tích hợp hai loop cho mỗi nhánh, chưa phải train nghiên cứu và không tính accuracy.

## Cách chạy

1. Import notebook này thành notebook mới trên Kaggle.
2. Add Input **cùng cache B gốc đã dùng thành công ở smoke trước**: `hypergamo-b-cache-355611502-b`, bên trong có `KLTN_HyperGAMO_B_50e988fae260` và đủ metadata/development. Không cần tải checkpoint model.
3. Bật GPU. Giữ nguyên cấu hình và chọn **Save Version → Save & Run All một lần**. Không chạy Run All trước rồi Save & Run All lần nữa.
4. Bốn worker chạy **tuần tự**, mỗi worker chỉ nhìn thấy một GPU. Deadline **30 phút là tổng cả bốn nhánh**, tính cả import/compile; không phải 30 phút mỗi nhánh. Hết giờ hoặc fail thì dừng, không tự chạy lại.
5. Tải `MOC2_IP_SMOKE_..._REPORT.zip` ở Output và gửi lại, kể cả FAILED/TIMEOUT. Packaging và Kaggle lưu/upload Output diễn ra sau worker; chúng không nằm trong thời gian tính toán GPU được cấp ở đây.

Code model/TensorFlow của notebook chưa được chạy ở máy cá nhân. Đã kiểm cú pháp và rà tĩnh; kết quả runtime cần đọc từ lượt Kaggle này.

## Phạm vi và khác biệt được công bố

- Giữ Indian Pines/B, patch11, PCA40, loss MSE, LR2e-4/beta1=0,5; V2 giữ rho=0,2 và k=3.
- Nhánh `original_gan` dùng head convex toàn support của B. ROS lấy real anchor; random-local và learned-local dùng cùng anchor/neighbor graph/rho, khác trọng số ngẫu nhiên và trọng số learned.
- C/D giữ Dropout rate0,5 và shape/scale; RNG được đổi thành **stateless route keys chung**. Mask thực dùng trong forward được ghi để kiểm. Đây là baseline của **runner ghép cặp mới**, không phải replay bitwise kết quả B cũ.
- Cả bốn nhánh phải khớp initialization C và trạng thái Adam C ban đầu, real batch/labels, fake-class schedule, vị trí C updates và **mask Dropout thực tế ở các C updates**.
- ROS/random-local/V2 khớp anchor; random-local/V2 còn khớp graph/neighbors. GAN gốc không có anchor nên không bị ép một anchor giả.
- Hai GAN khớp D/G/Dense-head initialization, noise, D targets, masks và vị trí cập nhật riêng G/D.


In [ ]:
# CẤU HÌNH — chỉ cần gắn Input B và bật GPU; thường không sửa các dòng này.
B_RUN = None               # tự tìm run B; hoặc đường dẫn thư mục run B trong /kaggle/input
GPU_INDEX = 0              # một GPU vật lý; worker remap thành /GPU:0
SEED = 42                  # khóa seed cho lượt kiểm, không tìm seed
TIME_LIMIT_SECONDS = 1800  # trần tổng bốn nhánh; không được tăng quá 30 phút


## Những gì notebook sẽ thực hiện

| Nhánh | Optimizer calls thật | C updates | Clock cuối |
|---|---:|---:|---:|
| Original GAN | 68 | 4 | 68 |
| Matched ROS | 4 | 4 | 68 |
| Random-local | 4 | 4 | 68 |
| Learned-local V2 | 68 | 4 | 68 |

Tổng **16 C updates**. Các fixture optimizer dùng biến scalar riêng được ghi tách, không tính là study updates.

- Xác minh 8 file/hash cache B trước import TF; chỉ đọc `X_train/y_train/coords_train`, không unpickle preprocessing hoặc materialize validation/test.
- NumPy metadata kiểm 90 loop: 45 batch/epoch, tail22, real order dùng lại, RNG riêng không làm lệch trace. **Các model update thật chỉ ở hai batch100 đầu; chưa cập nhật optimizer bằng tail22.**
- Mỗi loop có hai C updates. GAN thực hiện đủ 34 optimizer calls/loop. C-only tăng iteration-only ở các vị trí bỏ qua; kiểm model/moments không đổi, thêm fixture so Adam với reference có dummy-variable updates.
- Gradient observer ghi đúng compiled loss; gate freeze kiểm weights/BN/Adam m-v theo owner. Không ép moments C giống nhau sau khi nhận các input khác nhau.
- `consumed_trace.json` được ghi ở nơi các input thật được truyền vào routes. Launcher so hash và từng nhóm ghép cặp từ các bản ghi đó; fixture RNG riêng không đủ để PASS.
- Checkpoint chỉ kiểm roundtrip weights/BN/optimizer. Chưa kiểm continued trajectory hoặc exact Dropout resume; seed control bên ngoài checkpoint phải được dựng lại từ ledger trước cập nhật ở runner dài sau này.

Cell mã tiếp theo chỉ định nghĩa chuỗi source. Cell cuối khởi chạy suite trên Kaggle. Không cần upload thêm file Python.


In [ ]:
# Source worker tự chứa, chưa khởi chạy TF trong cell này.
WORKER_SOURCE = r'''"""Bounded Kaggle-only four-arm matched-runner integration gate.

No classification metrics or validation/test tensors. No campaign training.
This is a NEW matched RNG runner, not a bitwise replay of the historical B run.
"""
from __future__ import annotations
import gc
import hashlib
import importlib.metadata as metadata
import json
import math
import os
from pathlib import Path
import sys
import tempfile
import time
import traceback

np = tf = keras = None
CONFIG = REPORT = FOLDER = START = None
PAIRED_C_DROPOUT = PAIRED_D_DROPOUT = None
GAN_ARMS = ('original_gan','learned_local')
LOCAL_ARMS = ('random_local','learned_local')
ANCHOR_ARMS = ('matched_ros','random_local','learned_local')
LOOPS = 2
BATCH = 100

SHAPE = (11, 11, 40, 1)


FLAT = 4840


N_CLASSES = 16


K = 3


RHO = 0.2


RUN_ID = '50e988fae260'


IMPLEMENTATION = '5e15f8214bf273bb889187b96c80407a8000e5e1a3ef30734bcfa86ced275a6f'


EXPECTED_COUNTS = [17,602,341,110,167,301,13,182,8,375,1112,249,100,576,230,39]


PINNED_B = {
    'manifest.json':'41486ba39177e36f32e09c61cff870f4086a94c7b55f39db9dbe4b92093d6e67',
    'protocol_spec.json':'d4260da1ba9289a3183ab10905f995d6a2e8956b5651576734683017705bd202',
    'development/dev_split.npz':'1dcb4dac430f756d3bacc68d4e741bb5e34175ca018b744a6f3867e91b2cf608',
    'development/dev_split.npz.json':'72babb8a6056e3e298051a3581f03ae45256b556f11251a6377224ab354711b0',
    'development/dev_train_val.npz':'bf97ea110113ffd94afe1ba8a87d89a7ee3f5ad35efd3e3e214d5e300849fcc6',
    'development/dev_train_val.npz.json':'726cfc2135e316dd93f301dcc3babcd65e98ce0565d836470ede04c40ec02326',
    'development/preprocessing_dev.joblib':'d143a6d5f972fb07328e6c137d42a8332f91e06cbbbd96f41c8c5ea452a2dde1',
    'development/preprocessing_dev.joblib.json':'f5c80cf8a3e4f204c9caf15df948933466e6ca9028606c11cec8da9e0ebb6458',
}


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            deadline()
            h.update(chunk)
    return h.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)+'\n', encoding='utf-8')
    temporary.replace(path)


def deadline():
    if CONFIG is not None and time.monotonic() >= CONFIG['deadline_monotonic']:
        raise TimeoutError('Smoke exceeded the total 30-minute deadline, including startup/compile.')


def phase(name):
    deadline()
    REPORT['phase'] = name
    REPORT['elapsed_seconds'] = round(time.monotonic()-START, 3)
    atomic_json(FOLDER/'arm_report.json', REPORT)
    print(f"[{REPORT['elapsed_seconds']:.1f}s] {name}", flush=True)


def b_array_sha(value):
    # This is B's hash format, different from the smoke sha_array helper below.
    value = np.ascontiguousarray(value)
    h = hashlib.sha256()
    h.update(str(value.shape).encode())
    h.update(value.dtype.str.encode())
    h.update(value.tobytes())
    return h.hexdigest()


def load_input_verified():
    mode = CONFIG['data_mode']
    require(mode in ('auto_real_B', 'fixture'), 'Unknown data_mode')
    if mode == 'fixture':
        require(CONFIG['b_run'] is None, 'fixture mode must not also select a B run')
        rng = np.random.default_rng(19)
        y = np.concatenate([np.repeat(np.arange(15),4),np.repeat(15,6)]).astype(np.int64)
        X = rng.normal(0,.25,size=(len(y),*SHAPE)).astype(np.float32)
        return X,y,{'source':'synthetic_fixture','real_B_cache_verified':False,
                    'note':'Shape/wiring test only; not an Indian Pines data integration result.'}
    if CONFIG['b_run'] is not None:
        roots = [Path(CONFIG['b_run'])]
    else:
        preferred = Path('/kaggle/input/notebooks/minhlongg/notebook708cde343b')/f'KLTN_HyperGAMO_B_{RUN_ID}'
        roots = sorted(({preferred} if preferred.is_dir() else set()) | {
            p.parent.parent for p in Path('/kaggle/input').rglob('dev_train_val.npz')
            if p.parent.name == 'development' and p.parent.parent.name == f'KLTN_HyperGAMO_B_{RUN_ID}'
        })
    require(roots, 'Missing B output input. Add the original notebook Output containing KLTN_HyperGAMO_B_50e988fae260. No synthetic fallback was made.')
    valid, rejected = [], []
    for root in roots:
        deadline()
        try:
            actual = {}
            for relative, expected in PINNED_B.items():
                path = root/relative
                require(path.is_file(), f'Missing original B artifact: {path}')
                actual[relative] = file_sha(path)
                require(actual[relative] == expected, f'Original B hash mismatch: {path}')
            valid.append((root, actual))
        except RuntimeError as error:
            rejected.append(str(error))
    require(valid, 'No original B cache passed pinned hashes. '+ '; '.join(rejected))
    # Every eligible copy has the same eight pinned hashes, so sorted first is
    # deterministic even when multiple identical Kaggle input copies exist.
    root, actual = sorted(valid, key=lambda item: str(item[0]))[0]
    read_json = lambda relative: json.loads((root/relative).read_text(encoding='utf-8'))
    manifest = read_json('manifest.json')
    spec = read_json('protocol_spec.json')
    require(manifest['run_id'] == RUN_ID, 'Wrong B run_id')
    require(manifest['protocol_spec_sha256'] == actual['protocol_spec.json'], 'Broken B manifest/spec link')
    require(hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()[:12] == RUN_ID, 'Protocol/run-id hash mismatch')
    require(spec['implementation_sha256'] == IMPLEMENTATION and spec['patch_size'] == 11 and spec['pca_components'] == 40, 'Wrong B implementation/preprocessing config')
    for relative in ('development/dev_train_val.npz','development/dev_split.npz','development/preprocessing_dev.joblib'):
        sidecar = read_json(relative+'.json')
        require(sidecar['sha256'] == actual[relative], f'Broken sidecar hash: {relative}')
        require(sidecar['contract']['run_id'] == RUN_ID and sidecar['contract']['implementation'] == IMPLEMENTATION, f'Broken contract: {relative}')
    cache_contract = read_json('development/dev_train_val.npz.json')['contract']
    prep_contract = read_json('development/preprocessing_dev.joblib.json')['contract']
    require(cache_contract['kind'] == 'dev_cache', 'Not a development cache')
    require(cache_contract['preprocessing_sha256'] == actual['development/preprocessing_dev.joblib'], 'Cache/preprocessing hash mismatch')
    require(prep_contract['pca_components'] == 40 and prep_contract['pca_solver'] == 'full', 'Wrong PCA metadata')
    # Read only these three train arrays. Do not deserialize validation/test,
    # and never execute joblib/pickle preprocessing objects.
    with np.load(root/'development/dev_train_val.npz',allow_pickle=False) as data:
        X = data['X_train']
        y = data['y_train']
        coords = data['coords_train']
    require(X.dtype == np.float32 and X.shape == (4422,*SHAPE), 'Wrong development train feature dtype/shape')
    require(y.dtype == np.int64 and y.shape == (4422,), 'Wrong train label dtype/shape')
    require(coords.shape == (4422,2) and np.issubdtype(coords.dtype,np.integer), 'Wrong train center coordinates')
    require(np.all((coords>=0)&(coords<145)) and len(np.unique(coords,axis=0)) == 4422, 'Invalid train centers')
    require(np.all((y>=0)&(y<16)) and np.bincount(y,minlength=16).tolist() == EXPECTED_COUNTS, 'Wrong B per-class development counts')
    require(np.isfinite(X).all(), 'Nonfinite train features')
    require(cache_contract['train_coords'] == b_array_sha(coords), 'Train coordinate contract mismatch')
    require(prep_contract['fit_coords_sha256'] == b_array_sha(coords), 'PCA was not fit on these development train centers')
    return X,y,{'source':'original_B_development_train','run_path':str(root),
                'real_B_cache_verified':True,'pinned_file_hashes':actual,
                'eligible_identical_copies':len(valid),'rejected_candidates':rejected,
                'train_coords_B_hash':b_array_sha(coords),
                'validation_arrays_loaded':False,'test_arrays_loaded':False,
                'joblib_deserialized':False}


def load_runtime():
    global tf, keras
    tfv,kv = metadata.version('tensorflow'),metadata.version('tf-keras')
    require(tfv.split('.')[:2] == kv.split('.')[:2], f'TensorFlow/tf-keras minor mismatch: {tfv}/{kv}. No automatic install/retry.')
    import tensorflow as tensorflow
    import tf_keras as oldkeras
    tf,keras = tensorflow,oldkeras
    physical = tf.config.list_physical_devices('GPU')
    require(len(physical) == 1, f'Expected exactly one visible GPU; got {len(physical)}. CPU fallback is disabled.')
    tf.config.set_visible_devices(physical[:1], 'GPU')
    tf.config.experimental.set_memory_growth(physical[0], True)
    keras.mixed_precision.set_global_policy('float32')
    require(hasattr(keras.optimizers,'legacy') and hasattr(keras.optimizers.legacy,'Adam'), 'tf_keras legacy Adam is required')
    logical = tf.config.list_logical_devices('GPU')
    require(len(logical) == 1, 'Expected one logical GPU')
    with tf.device('/GPU:0'):
        probe = tf.linalg.matmul(tf.ones((2,2)),tf.ones((2,2)))
    require('GPU:0' in probe.device and np.isfinite(probe.numpy()).all(), 'GPU execution probe failed')
    return {'tensorflow':tf.__version__,'tf_keras':keras.__version__,
            'physical_visible':[p.name for p in physical], 'logical_visible':[p.name for p in logical],
            'cuda_visible_devices':os.environ['CUDA_VISIBLE_DEVICES'],
            'gpu_details':tf.config.experimental.get_device_details(physical[0]),
            'gpu_probe_device':probe.device,'cpu_fallback_allowed':False}


def sha_array(value):
    a = np.ascontiguousarray(value)
    h = hashlib.sha256()
    h.update(str(a.dtype).encode())
    h.update(str(a.shape).encode())
    h.update(a.tobytes())
    return h.hexdigest()


def build_graph_per_class(X, y):
    """Exact full-patch train-only kNN. Local supports sorted by global row order."""
    from scipy.spatial.distance import cdist
    counts = np.bincount(y, minlength=16)
    order = np.lexsort((np.arange(16),counts))
    if not np.all(counts[order[-1]] > counts[order[:-1]]) or np.any(counts[order[:-1]] < K+1):
        raise ValueError('Requires a strict single majority and >=4 train samples for each of 15 heads')
    inverse = np.empty(16, np.int64)
    inverse[order] = np.arange(16)
    y_sorted = inverse[y]
    supports, graph = [], []
    for h in range(15):
        support = np.asarray(X[y_sorted == h],np.float32)
        flat = support.reshape(len(support), FLAT).astype(np.float64)
        dist = cdist(flat, flat, metric='euclidean')
        np.fill_diagonal(dist,np.inf)
        # Tie-break by original train-row order (which is support-row order).
        nearest = np.empty((len(support),K),np.int32)
        for a in range(len(support)):
            nearest[a] = np.lexsort((np.arange(len(support)),dist[a]))[:K]
        assert not np.any(nearest == np.arange(len(support))[:,None])
        supports.append(support)
        graph.append(nearest)
    return {'supports':supports,'neighbor_graphs':graph,
            'counts':counts,'order':order,'inverse':inverse,'y_sorted':y_sorted}


def tao_classifier(shape, so_lop=16):
    L = keras.layers
    inp = L.Input(shape=shape)
    x = L.Conv3D(8, (1,1,7), activation="relu", padding="same")(inp)
    x = L.Conv3D(16, (3,3,5), activation="relu", padding="same")(x)
    x = L.Conv3D(32, (5,5,7), activation="relu", padding="same")(x)
    s = x.shape
    x = L.Reshape((s[1], s[2], s[3] * s[4]))(x)
    x = L.Conv2D(64, (3,3), activation="relu", padding="same")(x)
    x = L.GlobalAveragePooling2D()(x)
    x = PAIRED_C_DROPOUT(0.5)(x)
    out = L.Dense(so_lop, activation="softmax", kernel_initializer="he_normal")(x)
    return keras.Model(inp, out, name="classifier_3d_hypergamo")


def tao_generator(latent_dim, so_lop):
    L = keras.layers
    noise = L.Input(shape=(latent_dim,))
    labels = L.Input(shape=(so_lop,))
    x = L.Concatenate()([noise, labels])
    x = L.Dense(7*7*256, use_bias=False)(x)
    x = L.BatchNormalization(momentum=0.9)(x)
    x = L.LeakyReLU()(x)
    x = L.Reshape((7,7,256))(x)
    x = L.Conv2DTranspose(128, (5,5), strides=(1,1), padding="same", use_bias=False)(x)
    x = L.BatchNormalization(momentum=0.9)(x)
    x = L.LeakyReLU()(x)
    x = L.Conv2DTranspose(64, (5,5), strides=(2,2), padding="same", use_bias=False)(x)
    x = L.BatchNormalization(momentum=0.9)(x)
    x = L.LeakyReLU()(x)
    x = L.Conv2DTranspose(1, (5,5), strides=(2,2), padding="same", use_bias=False)(x)
    x = L.BatchNormalization(momentum=0.9)(x)
    out = L.Flatten()(x)  # 28×28 = 784
    return keras.Model([noise, labels], out, name="G")


def tao_discriminator(shape, so_lop):
    L = keras.layers
    inp = L.Input(shape=shape)
    x = L.Conv3D(8,(1,1,7),activation="relu",padding="same")(inp)
    x = L.Conv3D(16,(3,3,5),activation="relu",padding="same")(x)
    x = L.Conv3D(32,(5,5,7),activation="relu",padding="same")(x)
    s = x.shape
    x = L.Reshape((s[1],s[2],s[3]*s[4]))(x)
    x = L.Conv2D(64,(3,3),activation="relu",padding="same")(x)
    x = L.GlobalAveragePooling2D()(x)
    label = L.Input(shape=(so_lop,))
    x = L.Concatenate()([x,label])
    x = PAIRED_D_DROPOUT(.5)(x)
    out = L.Dense(1,activation="sigmoid",kernel_initializer="he_normal")(x)
    return keras.Model([inp,label],out,name="D")


def tao_head_v2(support, neighbors, rho=RHO):
    """B Dense(Nc) head, but local kNN masked softmax and a sampled anchor.

    Input 1: G output (batch,784). Input 2: head-local anchor row id (batch,1).
    The full Dense(Nc) remains trainable; no learned anchor choice and no
    learned rho. Supports/graph are fixed, non-trainable tensors.
    """
    assert support.ndim == 5 and support.shape[1:] == SHAPE
    n = len(support)
    assert neighbors.shape == (n,K)
    assert np.all((neighbors>=0) & (neighbors<n))
    assert not np.any(neighbors == np.arange(n)[:,None])
    L=keras.layers
    support_const=tf.constant(np.asarray(support.reshape(n,FLAT),np.float32))
    graph_const=tf.constant(neighbors,dtype=tf.int32)
    features=L.Input(shape=(784,),name='generator_features')
    anchor_ids=L.Input(shape=(1,),dtype='int32',name='head_local_anchor_id')
    # Crucial: Dense(Nc) has the SAME output size as the original B head.
    logits=L.Dense(n,activation=None,name='support_logits')(features)
    def local_mix(inputs):
        all_logits, ids=inputs
        ids=tf.reshape(tf.cast(ids,tf.int32),[-1])
        allowed=tf.gather(graph_const,ids)                    # (batch,3)
        allowed_logits=tf.gather(all_logits,allowed,axis=1,batch_dims=1)
        w=tf.nn.softmax(allowed_logits,axis=1)
        x0=tf.gather(support_const,ids)                       # (batch,4840)
        xs=tf.gather(support_const,allowed)                   # (batch,3,4840)
        return (1.0-rho)*x0 + rho*tf.einsum('bk,bkp->bp',w,xs)
    mixed=L.Lambda(local_mix,output_shape=(FLAT,),name='local_convex_mix')([logits,anchor_ids])
    output=L.Reshape(SHAPE,name='synthetic_patch')(mixed)
    return keras.Model([features,anchor_ids],output,name=f'head_v2_{n}')


def make_gradient_audit(owners):
    # Separate observational state: not an optimizer slot and not model weights.
    # Identity gradient_transformer observes actual compiled train_on_batch loss.
    names = list(owners)
    lookup = {}
    for i,name in enumerate(names):
        for variable in owners[name]['trainable']:
            require(variable.ref() not in lookup, 'A trainable variable has multiple owners')
            lookup[variable.ref()] = i
    with tf.device('/CPU:0'):
        audit = {'calls':tf.Variable(0,dtype=tf.int64,trainable=False),
                 'norm_sq':tf.Variable(np.zeros(len(names)),dtype=tf.float64,trainable=False),
                 'counts':tf.Variable(np.zeros(len(names),np.int32),trainable=False)}
    def observe(grads_and_vars):
        pairs = list(grads_and_vars)
        sums = [[] for _ in names]
        counts = [0]*len(names)
        finite_ops = []
        for gradient,variable in pairs:
            require(variable.ref() in lookup, 'Optimizer saw a variable with no registered owner')
            owner = lookup[variable.ref()]
            if gradient is None:
                continue
            require(not isinstance(gradient,tf.IndexedSlices), 'Unexpected sparse gradient in these Dense/Conv networks; stop rather than silently change sparse update semantics')
            finite_op = tf.debugging.assert_all_finite(gradient,'Nonfinite actual loss gradient')
            if finite_op is not None:
                finite_ops.append(finite_op)
            sums[owner].append(tf.reduce_sum(tf.square(tf.cast(gradient,tf.float64))))
            counts[owner] += 1
        with tf.control_dependencies(finite_ops):
            record_ops = [audit['norm_sq'].assign(tf.stack([tf.add_n(s) if s else tf.constant(0.,tf.float64) for s in sums])),
                          audit['counts'].assign(tf.constant(counts,tf.int32)),
                          audit['calls'].assign_add(1)]
        # Numerical identity; no clipping, resampling, extra forward/backward or
        # optimizer step. Control edges ensure the observation executes each call.
        with tf.control_dependencies(record_ops):
            return [(None if g is None else tf.identity(g),v) for g,v in pairs]
    return audit,observe


def state_snapshot(m):
    state = {}
    for name,owner in m['owners'].items():
        parts={part:[v.numpy().copy() for v in owner[part]] for part in ('trainable','nontrainable')}
        for slot in ('m','v'):
            values=[]
            for variable in owner['trainable']:
                value=m['opt'].get_slot(variable,slot)
                require(value is not None, f'Missing {slot} slot for {name}/{variable.name}')
                values.append(value.numpy().copy())
            parts[slot]=values
        for arrays in parts.values():
            require(all(np.isfinite(a).all() for a in arrays),f'Nonfinite state for {name}')
        state[name]=parts
    return state


def state_diff(before,after):
    require(before.keys() == after.keys(), 'Owner list changed')
    result={}
    for name in before:
        result[name]={}
        for part,arrays in before[name].items():
            other=after[name][part]
            require(len(arrays) == len(other),'Variable/slot count changed')
            changed=0;max_abs=0.
            for a,b in zip(arrays,other):
                require(a.shape == b.shape and a.dtype == b.dtype, 'Variable/slot signature changed')
                if not np.array_equal(a,b):
                    changed+=1
                    max_abs=max(max_abs,float(np.max(np.abs(a.astype(np.float64)-b))))
            result[name][part]={'changed_tensors':changed,'max_abs_delta':max_abs}
    return result


def fingerprint(m):
    h=hashlib.sha256()
    for name,parts in state_snapshot(m).items():
        for part,arrays in parts.items():
            h.update(f'{name}/{part}'.encode())
            for value in arrays:h.update(sha_array(value).encode())
    for value in m['opt'].get_weights():h.update(sha_array(value).encode())
    return h.hexdigest()


def update_gate(m,label,fn,active,family):
    phase(label)
    active=set(active)
    before=state_snapshot(m)
    clock=int(m['opt'].iterations.numpy())
    audit_calls=int(m['audit']['calls'].numpy())
    value=fn()
    loss=float(np.asarray(value).reshape(-1)[0])
    require(np.isfinite(loss),f'{label}: nonfinite loss')
    require(int(m['opt'].iterations.numpy()) == clock+1,f'{label}: shared Adam clock did not advance exactly once')
    require(int(m['audit']['calls'].numpy()) == audit_calls+1,f'{label}: actual compiled-loss gradient observer did not execute exactly once')
    counts=m['audit']['counts'].numpy().tolist()
    norm_sq=m['audit']['norm_sq'].numpy().tolist()
    gradients={}
    for i,(name,owner) in enumerate(m['owners'].items()):
        wanted=len(owner['trainable']) if name in active else 0
        require(counts[i] == wanted,f'{label}: optimizer gradient variable count for {name}: got {counts[i]}, expected {wanted}. Freeze/connectivity gate not met.')
        require(math.isfinite(norm_sq[i]) and norm_sq[i]>=0,f'{label}: invalid gradient norm')
        gradients[name]={'variable_count':counts[i],'l2_norm':math.sqrt(norm_sq[i]),
                         'numerically_active':bool(norm_sq[i]>0)}
    delta=state_diff(before,state_snapshot(m))
    for name,parts in delta.items():
        if name not in active:
            require(all(v['changed_tensors']==0 for v in parts.values()),f'{label}: frozen {name} weights/buffers/Adam slots changed')
        elif name != 'G':
            require(parts['nontrainable']['changed_tensors']==0,f'{label}: unexpected nontrainable buffer change for {name}')
    if family in ('C','D'):
        require(all(delta[name]['trainable']['changed_tensors']>0 and gradients[name]['numerically_active'] for name in active),f'{label}: no verified active classifier/discriminator update')
    evidence=all(gradients[name]['numerically_active'] and delta[name]['trainable']['changed_tensors']>0 for name in active)
    result={'route':label,'family':family,'loss':loss,'clock':clock+1,
            'actual_loss_gradients':gradients,'state_delta':delta,
            'active_trainable_update_observed':evidence,
            'frozen_weights_buffers_slots_unchanged':True}
    REPORT['routes'].append(result)
    phase(label+' complete')
    return result


def tao_head_convex(support, shape):
    """Tương đương denseGenProcessCreate: softmax weights × support matrix."""
    L = keras.layers
    support_flat = support.reshape(len(support), -1).astype(np.float32)
    support_const = tf.constant(support_flat)
    inp = L.Input(shape=(784,))
    weights = L.Dense(len(support), activation="softmax", name="support_weights")(inp)
    flat = L.Lambda(lambda w: tf.matmul(w, support_const), name="convex_sum")(weights)
    out = L.Reshape(shape)(flat)
    return keras.Model(inp, out, name=f"head_{len(support)}")
"""Pure NumPy input schedule for the bounded Indian Pines Moc 2 smoke.

This is a NEW keyed schedule, not a replay of the historical Protocol B GAN.
No TensorFlow, filesystem access, model construction, or optimization occurs here.
The runner must pass the returned arrays to its actual update calls and record
those inputs with ``consumed_trace``; a schedule fixture alone proves no training
route consumed the schedule.
"""

import hashlib
import json
from collections.abc import Mapping

import numpy as np


SCHEMA_VERSION = 1
N_CLASSES = 16
N_HEADS = 15
K_NEIGHBORS = 3

# Numeric domain tags are fixed protocol constants. Do not use Python hash().
_DOMAINS = {
    "real_order": 101,
    "fake_classes": 201,
    "c_anchors": 202,
    "c_noise": 203,
    "d_real_targets": 301,
    "d_fake_targets": 302,
    "local_random_weights": 401,
    "g_noise": 501,
    "g_anchors": 502,
}


def _require(condition, message):
    if not condition:
        raise ValueError(message)


def _integer(value, name, minimum=0):
    _require(isinstance(value, (int, np.integer)) and not isinstance(value, bool),
             f"{name} must be an integer")
    value = int(value)
    _require(value >= minimum, f"{name} must be >= {minimum}")
    return value


def _canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=True, allow_nan=False).encode("utf-8")


def array_sha(value):
    """Schedule-format digest, intentionally distinct from B's array hash."""
    value = np.ascontiguousarray(value)
    digest = hashlib.sha256()
    digest.update(value.dtype.str.encode("ascii"))
    digest.update(_canonical(list(value.shape)))
    digest.update(value.tobytes())
    return digest.hexdigest()


def _spec_sha(spec):
    return hashlib.sha256(_canonical({k: v for k, v in spec.items()
                                     if k != "schedule_sha256"})).hexdigest()


def _rng(spec, loop, domain, head=0):
    sequence = np.random.SeedSequence([
        SCHEMA_VERSION, int(spec["seed"]), int(loop), _DOMAINS[domain], int(head)
    ])
    # Pin the bit generator instead of depending on default_rng's future default.
    return np.random.Generator(np.random.PCG64(sequence))


def make_schedule(y_sorted, seed=42, batch_size=100, latent_dim=100,
                  g_points=7, n_heads=15, *, class_order):
    """Return a JSON-serializable schedule specification.

    ``y_sorted`` contains class-rank labels of the actual X_train row order.
    ``class_order`` is the corresponding 16 original 1-based class IDs, e.g.
    ``(prepared['order'] + 1).tolist()``. Support-row inventories are global
    X_train row indices in exactly the order used by ``X[y_sorted == h]``.
    """
    seed = _integer(seed, "seed")
    batch_size = _integer(batch_size, "batch_size", 1)
    latent_dim = _integer(latent_dim, "latent_dim", 1)
    g_points = _integer(g_points, "g_points", 1)
    n_heads = _integer(n_heads, "n_heads", 1)
    _require(n_heads == N_HEADS, "This smoke schedule is for 15 Indian Pines heads")
    labels = np.asarray(y_sorted)
    _require(labels.ndim == 1 and np.issubdtype(labels.dtype, np.integer),
             "y_sorted must be a one-dimensional integer array")
    _require(len(labels) > 0 and np.all((labels >= 0) & (labels < N_CLASSES)),
             "Invalid class-rank labels")
    labels = labels.astype(np.int64, copy=False)
    counts = np.bincount(labels, minlength=N_CLASSES)
    _require(np.all(counts > 0), "All 16 classes must be present")
    _require(np.all(np.diff(counts) >= 0), "y_sorted class counts must be sorted")
    _require(np.all(counts[-1] > counts[:-1]), "A strict single majority is required")
    _require(np.all(counts[:-1] >= K_NEIGHBORS + 1),
             "Every minority head needs at least four supports")
    order = np.asarray(class_order)
    _require(order.shape == (N_CLASSES,) and np.issubdtype(order.dtype, np.integer),
             "class_order must contain 16 original integer class IDs")
    _require(np.array_equal(np.sort(order), np.arange(1, N_CLASSES + 1)),
             "class_order must be a permutation of original IDs 1..16")
    spec = {
        "schema_version": SCHEMA_VERSION,
        "kind": "fresh_keyed_integration_schedule_not_historical_B_replay",
        "numpy_version": np.__version__,
        "bit_generator": "PCG64",
        "seed": seed,
        "samples": int(len(labels)),
        "batch_size": batch_size,
        "n_batches": int((len(labels) + batch_size - 1) // batch_size),
        "tail_batch_size": int((len(labels) - 1) % batch_size + 1),
        "latent_dim": latent_dim,
        "g_points": g_points,
        "n_classes": N_CLASSES,
        "n_heads": n_heads,
        "class_order_original_ids_1based": order.astype(np.int64).tolist(),
        "class_counts_sorted": counts.astype(np.int64).tolist(),
        "y_sorted": labels.tolist(),
        "support_row_ids": [np.flatnonzero(labels == h).tolist()
                            for h in range(N_CLASSES)],
        "domain_tags": dict(_DOMAINS),
        "real_order_policy": "one permutation reused at every wrap; no reshuffle",
        "g_pairing_policy": "GC and GD use the same per-head noise and anchors",
        "fake_class_policy": "minority probability proportional to majority-count deficit",
    }
    spec["real_order"] = _rng(spec, 0, "real_order").permutation(len(labels)).tolist()
    spec["schedule_sha256"] = _spec_sha(spec)
    validate_schedule(spec)
    return spec


def validate_schedule(spec):
    """Check a deserialized specification without generating or training a model."""
    _require(spec["schema_version"] == SCHEMA_VERSION, "Wrong schedule schema")
    _require(spec["numpy_version"] == np.__version__,
             "NumPy version changed; do not silently claim exact schedule replay")
    _require(spec["bit_generator"] == "PCG64" and spec["domain_tags"] == _DOMAINS,
             "RNG protocol changed")
    _require(spec["schedule_sha256"] == _spec_sha(spec), "Schedule digest mismatch")
    n = _integer(spec["samples"], "samples", 1)
    bs = _integer(spec["batch_size"], "batch_size", 1)
    _integer(spec["seed"], "seed")
    _integer(spec["latent_dim"], "latent_dim", 1)
    _integer(spec["g_points"], "g_points", 1)
    _require(spec["n_heads"] == 15 and spec["n_classes"] == 16,
             "Wrong class/head layout")
    y = np.asarray(spec["y_sorted"], dtype=np.int64)
    counts = np.asarray(spec["class_counts_sorted"], dtype=np.int64)
    _require(y.shape == (n,) and np.all((y >= 0) & (y < 16)), "Wrong label inventory")
    _require(counts.shape == (16,) and np.array_equal(np.bincount(y, minlength=16), counts),
             "Wrong support counts")
    _require(np.all(counts[:-1] >= 4) and np.all(np.diff(counts) >= 0)
             and np.all(counts[-1] > counts[:-1]), "Wrong sorted minority inventory")
    order = np.asarray(spec["real_order"], dtype=np.int64)
    _require(order.shape == (n,) and np.array_equal(np.sort(order), np.arange(n)),
             "real_order is not a complete permutation")
    _require(spec["n_batches"] == (n + bs - 1) // bs
             and spec["tail_batch_size"] == (n - 1) % bs + 1, "Wrong batch partition")
    original_ids = np.asarray(spec["class_order_original_ids_1based"])
    _require(original_ids.shape == (16,)
             and np.array_equal(np.sort(original_ids), np.arange(1, 17)),
             "Wrong original class order")
    _require(len(spec["support_row_ids"]) == 16, "Wrong support inventory length")
    for head in range(16):
        _require(np.array_equal(np.asarray(spec["support_row_ids"][head]),
                                np.flatnonzero(y == head)),
                 f"Support row order mismatch for class rank {head}")
    return True


def frame(spec, loop):
    """Generate the complete input frame for a zero-based real-batch loop.

    The arrays are generated by independent keyed domains; calls or extra draws
    in a control arm cannot advance another domain. ``g[h]`` must be used for
    BOTH GC_h and GD_h. This is a schedule contract, not evidence of consumption.
    """
    validate_schedule(spec)
    loop = _integer(loop, "loop")
    counts = np.asarray(spec["class_counts_sorted"], dtype=np.int64)
    y = np.asarray(spec["y_sorted"], dtype=np.int64)
    inventories = [np.asarray(rows, dtype=np.int64) for rows in spec["support_row_ids"]]
    batch_index = loop % spec["n_batches"]
    start = batch_index * spec["batch_size"]
    real_ids = np.asarray(spec["real_order"][start:start + spec["batch_size"]], dtype=np.int64)
    size, latent, gp = len(real_ids), spec["latent_dim"], spec["g_points"]
    deficits = counts[-1] - counts[:-1]
    cdf = np.cumsum(deficits / deficits.sum())
    cdf[-1] = 1.0
    fake_classes = np.searchsorted(cdf, _rng(spec, loop, "fake_classes").random(size),
                                   side="right").astype(np.int32)
    anchors = np.empty(size, dtype=np.int32)
    source_rows = np.empty(size, dtype=np.int64)
    for head in range(15):
        selected = np.flatnonzero(fake_classes == head)
        if len(selected):
            anchors[selected] = _rng(spec, loop, "c_anchors", head).integers(
                0, int(counts[head]), size=len(selected), dtype=np.int32)
            source_rows[selected] = inventories[head][anchors[selected]]
    fake_onehot = np.eye(16, dtype=np.float32)[fake_classes]
    g_inputs = []
    for head in range(15):
        head_anchors = _rng(spec, loop, "g_anchors", head).integers(
            0, int(counts[head]), size=(gp, 1), dtype=np.int32)
        condition = np.zeros((gp, 16), dtype=np.float32)
        condition[:, head] = 1.0
        g_inputs.append({
            "head": head,
            "noise": _rng(spec, loop, "g_noise", head).normal(size=(gp, latent)).astype(np.float32),
            "anchors": head_anchors,
            "support_row_ids": inventories[head][head_anchors[:, 0]],
            "condition": condition,
            "classifier_targets": 1.0 - condition,
            "discriminator_targets": np.ones((gp, 1), dtype=np.float32),
        })
    return {
        "loop": loop,
        "real_epoch": loop // spec["n_batches"],
        "batch_index": batch_index,
        "batch_size": size,
        "real_ids": real_ids,
        "real_classes": y[real_ids].astype(np.int32),
        "real_onehot": np.eye(16, dtype=np.float32)[y[real_ids]],
        "fake_classes": fake_classes,
        "fake_onehot": fake_onehot,
        "local_anchors": anchors,
        "support_row_ids": source_rows,
        "noise_c": _rng(spec, loop, "c_noise").normal(size=(size, latent)).astype(np.float32),
        "d_real_targets": (1.0 - _rng(spec, loop, "d_real_targets").uniform(
            0.0, 0.1, size=(size, 1))).astype(np.float32),
        "d_fake_targets": _rng(spec, loop, "d_fake_targets").uniform(
            0.0, 0.1, size=(size, 1)).astype(np.float32),
        "dirichlet_weights": _rng(spec, loop, "local_random_weights").dirichlet(
            np.ones(K_NEIGHBORS), size=size).astype(np.float32),
        "g": g_inputs,
    }


def _array_fields(value, prefix=""):
    result = {}
    if isinstance(value, np.ndarray):
        result[prefix] = value
    elif isinstance(value, Mapping):
        for key, child in value.items():
            result.update(_array_fields(child, f"{prefix}/{key}" if prefix else str(key)))
    elif isinstance(value, (list, tuple)):
        for index, child in enumerate(value):
            result.update(_array_fields(child, f"{prefix}/{index}" if prefix else str(index)))
    return result


def _field_records(fields):
    return {name: {"dtype": value.dtype.str, "shape": list(value.shape),
                   "sha256": array_sha(value)}
            for name, value in sorted(fields.items())}


def frame_digest(value):
    """Digest a frame's input arrays plus its loop/batch metadata."""
    payload = {"loop": int(value["loop"]), "real_epoch": int(value["real_epoch"]),
               "batch_index": int(value["batch_index"]), "batch_size": int(value["batch_size"]),
               "arrays": _field_records(_array_fields(value))}
    return hashlib.sha256(_canonical(payload)).hexdigest()


def consumed_trace(spec, loop, consumed, *, required_fields=None, label=""):
    """Validate explicitly supplied ACTUAL call inputs against the keyed frame.

    ``consumed`` may be nested like a frame or use flat paths, e.g.
    ``{'real_ids': ids_used, 'fake_classes': labels_used, 'g/0/noise': z_used}``.
    Values must be NumPy arrays of their original dtype, before TF conversion.
    ``required_fields`` should be supplied by each real route's adapter. This
    helper cannot by itself prove those arrays were passed to a model; invoke
    it where the runner actually performs that route, and retain route evidence.
    Different arms can compare per-field hashes for their shared inputs; an
    arm-specific label is intentionally excluded from the digest.
    """
    expected = _array_fields(frame(spec, loop))
    observed = _array_fields(consumed)
    _require(observed, "A consumed trace must contain actual input arrays")
    required = set(observed) if required_fields is None else set(required_fields)
    _require(required and required.issubset(observed), "Required consumed inputs are missing")
    _require(set(observed).issubset(expected), "Unknown consumed schedule fields")
    for name, value in observed.items():
        target = expected[name]
        _require(value.dtype == target.dtype and value.shape == target.shape,
                 f"Consumed dtype/shape mismatch: {name}")
        _require(np.array_equal(value, target), f"Actual input differs from schedule: {name}")
    records = _field_records(observed)
    payload = {"schedule_sha256": spec["schedule_sha256"], "loop": int(loop), "arrays": records}
    return {"label": str(label), **payload,
            "verified_fields": sorted(observed),
            "required_fields": sorted(required),
            "sha256": hashlib.sha256(_canonical(payload)).hexdigest(),
            "scope": "only explicitly supplied consumed inputs; model/update evidence is separate"}


def metadata_gate(spec, loops=90):
    """Bounded CPU metadata checks across two 45-batch IP epochs by default.

    This checks the schedule implementation, not training, accuracy, optimizer
    state, or an arm's actual consumption. The real runner must record those.
    """
    validate_schedule(spec)
    loops = _integer(loops, "loops", 1)
    _require(loops >= 2 * spec["n_batches"],
             "Metadata gate must cover at least two complete real-batch wraps")
    expected_counts = np.asarray(spec["class_counts_sorted"], dtype=np.int64)
    if spec["samples"] == 4422 and spec["batch_size"] == 100:
        _require(spec["n_batches"] == 45 and spec["tail_batch_size"] == 22,
                 "Indian Pines B development tail must be 22 at batch100")
    # A JSON roundtrip is part of the specification contract.
    restored = json.loads(json.dumps(spec, allow_nan=False))
    validate_schedule(restored)
    nuisance = np.random.default_rng(99991)
    digests, tails, real_rows, fake_counts = [], [], [], np.zeros(15, dtype=np.int64)
    for loop in range(loops):
        item = frame(spec, loop)
        digest = frame_digest(item)
        nuisance.normal(size=10000)  # unrelated arm/private draws
        _require(frame_digest(frame(restored, loop)) == digest,
                 "Unrelated RNG activity changed common frame")
        size = item["batch_size"]
        _require(item["real_ids"].shape == (size,) and item["noise_c"].shape == (size, spec["latent_dim"]),
                 "Wrong common input shape")
        _require(np.all((item["fake_classes"] >= 0) & (item["fake_classes"] < 15)),
                 "Fake schedule sampled majority/out-of-range class")
        _require(np.all(item["local_anchors"] >= 0)
                 and np.all(item["local_anchors"] < expected_counts[item["fake_classes"]]),
                 "Head-local C anchors are out of range")
        actual_labels = np.asarray(spec["y_sorted"])[item["support_row_ids"]]
        _require(np.array_equal(actual_labels, item["fake_classes"]),
                 "Source-row IDs do not belong to fake classes")
        _require(item["dirichlet_weights"].shape == (size, 3)
                 and np.all(item["dirichlet_weights"] >= 0)
                 and np.allclose(item["dirichlet_weights"].sum(axis=1), 1., rtol=0, atol=2e-7),
                 "Wrong random-local weights")
        _require(np.all((item["d_real_targets"] >= .9) & (item["d_real_targets"] <= 1.))
                 and np.all((item["d_fake_targets"] >= 0.) & (item["d_fake_targets"] <= .1)),
                 "Wrong D label smoothing")
        _require(len(item["g"]) == 15, "Missing per-head G inputs")
        for head, values in enumerate(item["g"]):
            _require(values["noise"].shape == (spec["g_points"], spec["latent_dim"])
                     and values["anchors"].shape == (spec["g_points"], 1),
                     "Wrong G input shape")
            _require(np.all((values["anchors"] >= 0) & (values["anchors"] < expected_counts[head])),
                     "Head-local G anchor is out of range")
            _require(np.all(np.asarray(spec["y_sorted"])[values["support_row_ids"]] == head),
                     "G source-row inventory is wrong")
        if item["batch_index"] == spec["n_batches"] - 1:
            tails.append({"loop": loop, "size": size})
            _require(size == spec["tail_batch_size"], "Wrong real-batch tail")
        real_rows.append(item["real_ids"])
        fake_counts += np.bincount(item["fake_classes"], minlength=15)
        digests.append(digest)
    all_real = np.concatenate(real_rows)
    for epoch in range(loops // spec["n_batches"]):
        first = epoch * spec["samples"]
        _require(np.array_equal(all_real[first:first + spec["samples"]], spec["real_order"]),
                 "Real order changed or missed samples at wrap")
    return {
        "pass": True,
        "scope": "pure NumPy metadata only; no model or actual training-route evidence",
        "schema_version": SCHEMA_VERSION,
        "schedule_sha256": spec["schedule_sha256"],
        "seed": spec["seed"],
        "numpy_version": np.__version__,
        "class_order_original_ids_1based": spec["class_order_original_ids_1based"],
        "samples": spec["samples"], "n_batches": spec["n_batches"],
        "tail_batch_size": spec["tail_batch_size"], "loops_checked": loops,
        "tail_loops": tails, "real_presentations_checked": int(len(all_real)),
        "fake_presentations_by_head": fake_counts.tolist(),
        "frame_trace_sha256": hashlib.sha256(_canonical(digests)).hexdigest(),
        "json_roundtrip_checked": True,
        "unrelated_rng_invariance_checked": True,
        "historical_B_replay_claimed": False,
        "model_training_performed": False,
    }



def json_sha(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),ensure_ascii=True,allow_nan=False).encode()).hexdigest()


def arrays_sha(arrays):
    return json_sha([sha_array(np.asarray(a)) for a in arrays])


def dropout_key(seed,loop,route,head=-1):
    payload=json.dumps([int(seed),int(loop),str(route),int(head)],separators=(',',':')).encode()
    return np.frombuffer(hashlib.sha256(payload).digest()[:8],dtype='<i4').copy()


def recorded_dropout_factory(width,owner):
    """Rate-0.5 Dropout with external stateless seed and actual mask recorder.

    Control/audit variables are intentionally separate from frozen network
    weights and Adam state. They are reconstructed from route keys, not treated
    as model parameters. The layer must have no weights of its own.
    """
    with tf.device('/CPU:0'):
        control={'seed':tf.Variable([0,1],dtype=tf.int32,trainable=False),
                 'used_seed':tf.Variable([0,0],dtype=tf.int32,trainable=False),
                 'calls':tf.Variable(0,dtype=tf.int64,trainable=False),
                 'rows':tf.Variable(0,dtype=tf.int32,trainable=False),
                 'mask':tf.Variable(np.zeros((BATCH,width),np.int32),trainable=False)}
    class RecordedDropout(keras.layers.Layer):
        def __init__(self,rate=.5,**kwargs):
            super().__init__(**kwargs)
            require(float(rate)==.5,'Matched runner keeps Dropout rate 0.5')
            self.rate=float(rate)
        def call(self,inputs,training=None):
            def dropped():
                seed=control['seed'].read_value()
                rows=tf.shape(inputs)[0]
                checks=[tf.debugging.assert_less_equal(rows,BATCH),
                        tf.debugging.assert_equal(tf.shape(inputs)[1],width)]
                with tf.control_dependencies([c for c in checks if c is not None]):
                    mask=tf.cast(tf.random.stateless_uniform(tf.shape(inputs),seed=seed,dtype=tf.float32)>=self.rate,tf.int32)
                operations=[control['mask'].assign(tf.pad(mask,[[0,BATCH-rows],[0,0]])),
                            control['rows'].assign(rows),control['used_seed'].assign(seed),
                            control['calls'].assign_add(1)]
                with tf.control_dependencies(operations):
                    return inputs*tf.cast(mask,inputs.dtype)/(1.-self.rate)
            return keras.backend.in_train_phase(dropped,inputs,training=training)
        def get_config(self):
            return {**super().get_config(),'rate':self.rate}
    RecordedDropout.__name__='RecordedDropout_'+owner
    return RecordedDropout,control


def build_models(X,y,arm,seed):
    global PAIRED_C_DROPOUT,PAIRED_D_DROPOUT
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)
    prepared=build_graph_per_class(X,y)
    context={}
    def observer(pairs):
        require('observe' in context,'Optimizer observation not initialized')
        return context['observe'](pairs)
    opt=keras.optimizers.legacy.Adam(learning_rate=2e-4,beta_1=.5,gradient_transformers=[observer])
    PAIRED_C_DROPOUT,c_control=recorded_dropout_factory(64,'C')
    keras.utils.set_random_seed(seed)
    C=tao_classifier(SHAPE,16)
    C.compile(loss='mean_squared_error',optimizer=opt)
    controls={'C':c_control}
    networks={'C':C}
    heads=[];vs_c=[];vs_d=[]
    if arm in GAN_ARMS:
        PAIRED_D_DROPOUT,d_control=recorded_dropout_factory(80,'D')
        controls['D']=d_control
        keras.utils.set_random_seed(seed+101)
        D=tao_discriminator(SHAPE,16)
        D.compile(loss='mean_squared_error',optimizer=opt)
        keras.utils.set_random_seed(seed+102)
        G=tao_generator(100,16)
        for h,(support,graph) in enumerate(zip(prepared['supports'],prepared['neighbor_graphs'])):
            keras.utils.set_random_seed(seed+1000+h)
            heads.append(tao_head_convex(support,SHAPE) if arm=='original_gan' else tao_head_v2(support,graph))
        C.trainable=False;D.trainable=False
        for h,head in enumerate(heads):
            z=keras.layers.Input(shape=(100,));condition=keras.layers.Input(shape=(16,))
            if arm=='original_gan':
                fake=head(G([z,condition]));inputs=[z,condition]
            else:
                anchor=keras.layers.Input(shape=(1,),dtype='int32')
                fake=head([G([z,condition]),anchor]);inputs=[z,condition,anchor]
            mc=keras.Model(inputs,C(fake),name=f'GC_{h}')
            mc.compile(loss='mean_squared_error',optimizer=opt)
            vs_c.append(mc)
            z2=keras.layers.Input(shape=(100,));cond2=keras.layers.Input(shape=(16,));cond3=keras.layers.Input(shape=(16,))
            if arm=='original_gan':
                fake2=head(G([z2,cond2]));inputs2=[z2,cond2,cond3]
            else:
                anchor2=keras.layers.Input(shape=(1,),dtype='int32')
                fake2=head([G([z2,cond2]),anchor2]);inputs2=[z2,cond2,cond3,anchor2]
            md=keras.Model(inputs2,D([fake2,cond3]),name=f'GD_{h}')
            md.compile(loss='mean_squared_error',optimizer=opt)
            vs_d.append(md)
        C.trainable=True;D.trainable=True
        networks.update(D=D,G=G)
        networks.update({f'head_{h}':head for h,head in enumerate(heads)})
    owners={name:{'trainable':tuple(model.trainable_variables),'nontrainable':tuple(model.non_trainable_variables)} for name,model in networks.items()}
    all_variables=[v for owner in owners.values() for v in owner['trainable']]
    require(all('GPU:0' in v.device for v in all_variables),'Unexpected non-GPU model variables')
    opt._create_all_weights(all_variables)
    require(set(opt.get_slot_names())=={'m','v'},'Unexpected shared Adam slots')
    model_refs={v.ref() for model in networks.values() for v in model.weights}
    for name,control in controls.items():
        require(not any(v.ref() in model_refs for v in control.values()),f'{name} dropout control was accidentally tracked as network weights')
        for layer in networks[name].layers:
            if layer.__class__.__name__.startswith('RecordedDropout'):
                require(not layer.weights,'RecordedDropout must not own trainable/frozen model state')
    audit,observe=make_gradient_audit(owners)
    context['observe']=observe
    return {**networks,'C':C,'heads':heads,'vs_c':vs_c,'vs_d':vs_d,
            'opt':opt,'owners':owners,'audit':audit,'controls':controls,**prepared}


def initial_fingerprints(m):
    def trainable(name):return arrays_sha(v.numpy() for v in m['owners'][name]['trainable'])
    result={'C_trainable_sha256':trainable('C'),
            'C_nontrainable_sha256':arrays_sha(v.numpy() for v in m['owners']['C']['nontrainable']),
            'C_adam_slots_sha256':arrays_sha(m['opt'].get_slot(v,s).numpy() for v in m['owners']['C']['trainable'] for s in ('m','v'))}
    if CONFIG['arm'] in GAN_ARMS:
        result.update(D_trainable_sha256=trainable('D'),G_trainable_sha256=trainable('G'),
                      head_dense_trainable_sha256=arrays_sha(v.numpy() for head in m['heads'] for v in head.trainable_variables))
    return result


def dropout_unit_gate(m):
    phase('stateless Dropout mask gate: main C / tail / private G sizes')
    before=fingerprint(m)
    control=m['controls']['C']
    layer=next(layer for layer in m['C'].layers if layer.__class__.__name__.startswith('RecordedDropout'))
    hashes={}
    for rows in (100,22,7):
        x=tf.ones((rows,64),tf.float32)
        key=dropout_key(CONFIG['seed'],0,'C_real')
        control['seed'].assign(key)
        first=layer(x,training=True).numpy()
        require(np.all((first==0)|(first==2)),'Dropout rate/scale test failed')
        control['seed'].assign(dropout_key(CONFIG['seed'],12,'GC',4))
        layer(x,training=True)
        control['seed'].assign(key)
        second=layer(x,training=True).numpy()
        require(np.array_equal(first,second),'Private GAN dropout key shifted common C mask')
        require(np.array_equal(layer(x,training=False).numpy(),np.ones((rows,64),np.float32)),'Dropout changed inference')
        hashes[str(rows)]=sha_array(first)
    require(fingerprint(m)==before,'Dropout-only gate changed model/Adam state')
    return {'pass':True,'same_key_after_private_key':True,'batch_mask_hashes':hashes,
            'rate':.5,'RNG_policy':'stateless keys, new matched runner, not historical B replay'}


def clock_skip_fixture():
    """Reference actual legacy Adam against iteration-only skips on C state.

    Dummy updates use a separate dummy variable in the REFERENCE optimizer;
    never a zero-gradient update on the classifier. Not study-model updates.
    """
    phase('scalar shared-Adam skip/reference fixture (not study updates)')
    a=tf.Variable([.5],dtype=tf.float32);b=tf.Variable([.5],dtype=tf.float32);dummy=tf.Variable([.2],dtype=tf.float32)
    actual=keras.optimizers.legacy.Adam(learning_rate=2e-4,beta_1=.5)
    reference=keras.optimizers.legacy.Adam(learning_rate=2e-4,beta_1=.5)
    actual._create_all_weights([a]);reference._create_all_weights([b,dummy])
    def values(opt,variable):return [variable.numpy().copy()]+[opt.get_slot(variable,s).numpy().copy() for s in ('m','v')]
    for loop in range(2):
        for grad,skip in ((.2,1),(-.3,31)):
            actual.apply_gradients([(tf.constant([grad],tf.float32),a)])
            reference.apply_gradients([(tf.constant([grad],tf.float32),b)])
            require(all(np.array_equal(x,y) for x,y in zip(values(actual,a),values(reference,b))),'Adam skip/reference C state mismatch')
            before=values(actual,a)
            actual.iterations.assign_add(skip)
            require(all(np.array_equal(x,y) for x,y in zip(before,values(actual,a))),'Iteration-only skip changed C weights/moments')
            for _ in range(skip):
                reference.apply_gradients([(tf.constant([.1],tf.float32),dummy)])
            require(int(actual.iterations.numpy())==int(reference.iterations.numpy()),'Fixture clock mismatch')
    return {'pass':True,'reference_iterations':68,'actual_C_applications':4,
            'reference_C_applications':4,'reference_dummy_applications':64,
            'scope':'optimizer-only scalar fixture; no classifier and no accuracy'}


def skip_iterations(m,amount,label):
    phase(label)
    before=state_snapshot(m);clock=int(m['opt'].iterations.numpy());calls=int(m['audit']['calls'].numpy())
    m['opt'].iterations.assign_add(amount)
    delta=state_diff(before,state_snapshot(m))
    require(all(v['changed_tensors']==0 and v['max_abs_delta']==0 for parts in delta.values() for v in parts.values()),'Skipped optimizer positions changed model or m/v')
    require(int(m['opt'].iterations.numpy())==clock+amount and int(m['audit']['calls'].numpy())==calls,'Iteration-only skip called optimizer or wrong clock')
    REPORT['iteration_skips'].append({'label':label,'amount':amount,'clock_before':clock,'clock_after':clock+amount,'weights_m_v_unchanged':True})


def controlled_update(m,model_owner,route,loop,head,fn,active,family):
    control=m['controls'][model_owner]
    seed=dropout_key(CONFIG['seed'],loop,route,head)
    control['seed'].assign(seed)
    calls=int(control['calls'].numpy())
    before=int(m['opt'].iterations.numpy())
    result=update_gate(m,f'loop{loop}/{route}'+(f'_{head}' if head>=0 else ''),fn,active,family)
    require(int(control['calls'].numpy())==calls+1,f'{route}: actual Dropout recorder did not execute once')
    require(np.array_equal(control['used_seed'].numpy(),seed),f'{route}: stale Dropout seed captured by train function')
    rows=int(control['rows'].numpy())
    record={'loop':loop,'route':route,'head':head,'clock_before':before,'clock_after':int(m['opt'].iterations.numpy()),
            'dropout_seed':seed.tolist(),'dropout_mask_sha256':sha_array(control['mask'].numpy()[:rows]),'rows':rows}
    return result,record


def infer_fake(m,item,arm,spec):
    """Return patches and evidence derived from arrays actually used to render."""
    before=fingerprint(m);clock=int(m['opt'].iterations.numpy())
    labels=item['fake_onehot'].copy()
    actual_classes=labels.argmax(1).astype(np.int32)
    anchors=item['local_anchors'].copy()
    fake=np.empty((len(labels),*SHAPE),np.float32)
    source_rows=np.empty(len(labels),np.int64)
    neighbors_local=np.empty((len(labels),3),np.int32)
    neighbors_global=np.empty((len(labels),3),np.int64)
    used_anchors=np.empty(len(labels),np.int32)
    used_weights=np.empty((len(labels),3),np.float32)
    features=None
    if arm in GAN_ARMS:
        noise=item['noise_c'].copy()
        features=m['G'].predict_on_batch([noise,labels])
    for h,support in enumerate(m['supports']):
        ii=np.flatnonzero(actual_classes==h)
        if not len(ii):continue
        if arm=='original_gan':
            fake[ii]=m['heads'][h].predict_on_batch(features[ii])
            continue
        local=anchors[ii]
        used_anchors[ii]=local
        inventory=np.asarray(spec['support_row_ids'][h],np.int64)
        source_rows[ii]=inventory[local]
        if arm=='matched_ros':
            fake[ii]=support[local]
        else:
            allowed=m['neighbor_graphs'][h][local]
            neighbors_local[ii]=allowed
            neighbors_global[ii]=inventory[allowed]
            if arm=='random_local':
                weights=item['dirichlet_weights'][ii].copy()
                used_weights[ii]=weights
                fake[ii]=(1-RHO)*support[local]+RHO*np.einsum('bk,bk...->b...',weights,support[allowed])
            else:
                fake[ii]=m['heads'][h].predict_on_batch([features[ii],local[:,None]])
            movement=np.linalg.norm((fake[ii]-support[local]).reshape(len(ii),-1).astype(np.float64),axis=1)
            neighbor_distance=np.linalg.norm((support[allowed]-support[local,None]).reshape(len(ii),3,-1).astype(np.float64),axis=2)
            bound=RHO*neighbor_distance.max(axis=1)
            require(np.all(movement<=bound+2e-5*(1+bound)),'Local mixing violated the fixed rho displacement bound')
    require(np.isfinite(fake).all(),'Nonfinite fake patches')
    require(fingerprint(m)==before and int(m['opt'].iterations.numpy())==clock,'Fake-slot inference changed weights/BN/Adam state')
    evidence={'fake_classes':actual_classes,'fake_onehot':labels}
    if arm in ANCHOR_ARMS:
        evidence.update(local_anchors=used_anchors,support_row_ids=source_rows)
    if arm in LOCAL_ARMS:
        evidence.update(neighbors_local=neighbors_local,neighbors_global=neighbors_global)
    if arm in GAN_ARMS:evidence['noise_c']=noise
    if arm=='random_local':evidence['dirichlet_weights']=used_weights
    return fake,labels,evidence


def checkpoint_object(m):
    values={'C':m['C'],'optimizer':m['opt']}
    if CONFIG['arm'] in GAN_ARMS:
        values.update(D=m['D'],G=m['G'])
        values.update({f'head_{h:02d}':head for h,head in enumerate(m['heads'])})
    return tf.train.Checkpoint(**values)


def run_arm():
    phase('verify original B provenance before TensorFlow')
    X,y,source=load_input_verified()
    require(source['real_B_cache_verified'],'Moc2 accepts verified B cache only')
    REPORT['input']=source
    REPORT['input'].update(shape=list(X.shape),X_sha256=sha_array(X),y_sha256=sha_array(y))
    phase('TensorFlow / one GPU')
    REPORT['runtime']=load_runtime()
    arm=CONFIG['arm'];seed=CONFIG['seed']
    if arm not in GAN_ARMS:REPORT['clock_fixture']=clock_skip_fixture()
    phase('build fresh component-keyed models and train-only graph')
    with tf.device('/GPU:0'):
        m=build_models(X,y,arm,seed)
    REPORT['initialization']=initial_fingerprints(m)
    REPORT['dropout_unit_gate']=dropout_unit_gate(m)
    spec=make_schedule(m['y_sorted'],seed=seed,batch_size=BATCH,latent_dim=100,g_points=7,n_heads=15,class_order=(m['order']+1).tolist())
    atomic_json(FOLDER/'schedule_spec.json',spec)
    phase('CPU metadata trace: 90 loops, tail22 and real-order wrap; no model updates')
    REPORT['schedule_metadata']=metadata_gate(spec,loops=90)
    ledger={'attempt_id':CONFIG['attempt_id'],'arm':arm,'common_c':[],'anchors':[],
            'local_geometry':[],'common_gan':[]}
    graph_hashes=[sha_array(g) for g in m['neighbor_graphs']]
    for loop in range(LOOPS):
        phase(f'loop {loop}: consume keyed frame, batch100')
        item=frame(spec,loop)
        t=34*loop
        require(int(m['opt'].iterations.numpy())==t,'Wrong loop start clock')
        ids=item['real_ids'].copy();real=X[ids].copy();real_labels=item['real_onehot'].copy()
        consumed=consumed_trace(spec,loop,{'real_ids':ids,'real_onehot':real_labels},required_fields=['real_ids','real_onehot'],label='C_real')
        _,c_record=controlled_update(m,'C','C_real',loop,-1,lambda:m['C'].train_on_batch(real,real_labels),{'C'},'C')
        c_record.update(input_real_sha256=sha_array(real),labels_sha256=sha_array(real_labels),schedule=consumed)
        c_record['real_train_row_ids']=ids.tolist()
        ledger['common_c'].append(c_record)
        if arm in GAN_ARMS:
            targets=item['d_real_targets'].copy()
            consumed=consumed_trace(spec,loop,{'real_ids':ids,'real_onehot':real_labels,'d_real_targets':targets},required_fields=['real_ids','real_onehot','d_real_targets'],label='D_real')
            _,record=controlled_update(m,'D','D_real',loop,-1,lambda:m['D'].train_on_batch([real,real_labels],targets),{'D'},'D')
            record['schedule']=consumed;ledger['common_gan'].append(record)
        else:
            skip_iterations(m,1,f'loop{loop}/skip_D_real')
        fake,fake_labels,evidence=infer_fake(m,item,arm,spec)
        REPORT['fake_patch_hashes'].append(sha_array(fake))
        if arm in GAN_ARMS:
            consumed=consumed_trace(spec,loop,{'noise_c':evidence['noise_c'],'fake_onehot':fake_labels},required_fields=['noise_c','fake_onehot'],label='G_fake_inference')
            ledger['common_gan'].append({'loop':loop,'route':'G_fake_inference','clock':t+2,'schedule':consumed})
        if arm in ANCHOR_ARMS:
            consumed=consumed_trace(spec,loop,{key:evidence[key] for key in ('fake_classes','local_anchors','support_row_ids')},required_fields=['fake_classes','local_anchors','support_row_ids'],label='actual_C_fake_anchors')
            ledger['anchors'].append({'schedule':consumed,'local_anchor_ids':evidence['local_anchors'].tolist(),
                                      'global_train_row_ids':evidence['support_row_ids'].tolist(),
                                      'class_ranks':evidence['fake_classes'].tolist()})
        if arm in LOCAL_ARMS:
            ledger['local_geometry'].append({'loop':loop,'k':K,'rho':RHO,'graph_hashes':graph_hashes,
                                            'neighbors_local_sha256':sha_array(evidence['neighbors_local']),
                                            'neighbors_global_sha256':sha_array(evidence['neighbors_global']),
                                            'neighbors_local_ids':evidence['neighbors_local'].tolist(),
                                            'neighbors_global_train_row_ids':evidence['neighbors_global'].tolist()})
        if arm=='random_local':
            REPORT['random_weight_consumption'].append(consumed_trace(spec,loop,{'dirichlet_weights':evidence['dirichlet_weights']},required_fields=['dirichlet_weights'],label='actual_random_local_weights'))
        consumed=consumed_trace(spec,loop,{'fake_classes':evidence['fake_classes'],'fake_onehot':fake_labels},required_fields=['fake_classes','fake_onehot'],label='C_fake')
        _,c_record=controlled_update(m,'C','C_fake',loop,-1,lambda:m['C'].train_on_batch(fake,fake_labels),{'C'},'C')
        c_record.update(labels_sha256=sha_array(fake_labels),schedule=consumed)
        c_record['fake_class_ranks']=evidence['fake_classes'].tolist()
        ledger['common_c'].append(c_record)
        if arm in GAN_ARMS:
            targets=item['d_fake_targets'].copy()
            consumed=consumed_trace(spec,loop,{'fake_onehot':fake_labels,'d_fake_targets':targets},required_fields=['fake_onehot','d_fake_targets'],label='D_fake')
            _,record=controlled_update(m,'D','D_fake',loop,-1,lambda:m['D'].train_on_batch([fake,fake_labels],targets),{'D'},'D')
            record['schedule']=consumed;ledger['common_gan'].append(record)
            for h,g_inputs in enumerate(item['g']):
                noise=g_inputs['noise'].copy();cond=g_inputs['condition'].copy();anchor=g_inputs['anchors'].copy()
                gc_target=g_inputs['classifier_targets'].copy();gd_target=g_inputs['discriminator_targets'].copy()
                gc_args=[noise,cond] if arm=='original_gan' else [noise,cond,anchor]
                gd_args=[noise,cond,cond] if arm=='original_gan' else [noise,cond,cond,anchor]
                if arm=='learned_local':
                    REPORT['G_anchor_consumption'].append(consumed_trace(spec,loop,{f'g/{h}/anchors':gc_args[-1]},required_fields=[f'g/{h}/anchors'],label=f'GC_GD_anchor_{h}'))
                    require(np.array_equal(gc_args[-1],gd_args[-1]),'GC/GD did not consume the same private anchor IDs')
                for route,args,target,family,model in (('GC',gc_args,gc_target,'GC',m['vs_c'][h]),('GD',gd_args,gd_target,'GD',m['vs_d'][h])):
                    fields={f'g/{h}/noise':args[0],f'g/{h}/condition':args[1],f'g/{h}/'+('classifier_targets' if route=='GC' else 'discriminator_targets'):target}
                    consumed=consumed_trace(spec,loop,fields,required_fields=list(fields),label=f'{route}_{h}')
                    _,record=controlled_update(m,'C' if route=='GC' else 'D',route,loop,h,lambda model=model,args=args,target=target:model.train_on_batch(args,target),{'G',f'head_{h}'},family)
                    if route=='GD':
                        require(np.array_equal(args[1],args[2]),'G-vs-D class conditions differ')
                        record['D_condition_sha256']=sha_array(args[2])
                    record['schedule']=consumed;ledger['common_gan'].append(record)
        else:
            skip_iterations(m,31,f'loop{loop}/skip_D_fake_and_all_G')
        require(int(m['opt'].iterations.numpy())==t+34,'Wrong loop-end shared Adam clock')
        atomic_json(FOLDER/'consumed_trace.json',ledger)
    applications=68 if arm in GAN_ARMS else 4
    require(len(REPORT['routes'])==applications and int(m['audit']['calls'].numpy())==applications,'Unexpected actual optimizer application count')
    require(int(m['opt'].iterations.numpy())==68,'Final clock must be 68')
    if arm in GAN_ARMS:
        # The loop variable otherwise retains the old compiled composite while
        # rebuilding for checkpoint restore, keeping its G/D/optimizer alive.
        del model
        for family in ('GC','GD'):
            verified=[r for r in REPORT['routes'] if r['family']==family and r['active_trainable_update_observed']]
            require(verified,f'No nonzero G/head updates under {family}')
        for h in range(15):
            require(any(r['family'] in ('GC','GD') and r['route'].endswith('_'+str(h)) and r['active_trainable_update_observed'] for r in REPORT['routes']),f'Head {h} update evidence unresolved')
    REPORT['trace']={'common_C':json_sha(ledger['common_c']),
                     'anchors_C':json_sha(ledger['anchors']) if arm in ANCHOR_ARMS else None,
                     'local_geometry':json_sha(ledger['local_geometry']) if arm in LOCAL_ARMS else None,
                     'GAN_private':json_sha(ledger['common_gan']) if arm in GAN_ARMS else None,
                     'consumed_loops':LOOPS}
    REPORT['clocks']={'C_updates':4,'final_clock':68,'real_optimizer_calls':applications}
    phase('checkpoint model/BN/Adam roundtrip; not trajectory continuation')
    before=fingerprint(m)
    with tempfile.TemporaryDirectory(prefix='moc2_ckpt_') as temporary:
        prefix=str(Path(temporary)/'checkpoint')
        checkpoint_object(m).write(prefix)
        del m
        keras.backend.clear_session();gc.collect();deadline()
        with tf.device('/GPU:0'):
            m=build_models(X,y,arm,seed)
        checkpoint_object(m).read(prefix).assert_consumed()
        require(fingerprint(m)==before and int(m['opt'].iterations.numpy())==68,'Checkpoint state roundtrip failed')
    REPORT['checkpoint']={'weights_BN_Adam_roundtrip':True,'optimizer_iterations':68,'fingerprint':before,
                          'exact_resume_or_continuation_tested':False,'external_dropout_control_roundtrip_tested':False,
                          'external_dropout_seed_policy':'Recompute from seed/loop/route/head before each update; not exercised after restore in this gate'}
    del m
    keras.backend.clear_session();gc.collect();deadline()
    REPORT['gates']={'freeze':True,'trace_consumption':True,'dropout_C_mask_keys':True,
                     'iteration_skips':True,'checkpoint_roundtrip':True}
    REPORT['iteration_skip_scope']=('not applicable: all 68 optimizer positions are actual updates' if arm in GAN_ARMS
                                    else 'four iteration-only skip blocks verified; scalar reference fixture passed')
    REPORT.update(status='PASS',**{'pass':True},phase='complete',elapsed_seconds=round(time.monotonic()-START,3))
    atomic_json(FOLDER/'arm_report.json',REPORT)
    print('ARM PASS:',arm,'| 4 C updates, clock68 | no accuracy/test',flush=True)


def main():
    global CONFIG,REPORT,FOLDER,START,np
    CONFIG=json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
    FOLDER=Path(CONFIG['attempt_dir']);START=CONFIG['start_monotonic']
    REPORT={'attempt_id':CONFIG['attempt_id'],'arm':CONFIG['arm'],'config_sha256':CONFIG['config_sha256'],
            'worker_sha256':CONFIG['worker_sha256'],'status':'RUNNING','pass':False,
            'scope':'two-loop four-arm runner integration, not classification study',
            'rho':RHO,'k':K,'seed':CONFIG['seed'],'batch_size':BATCH,'loops':LOOPS,
            'routes':[],'iteration_skips':[],'fake_patch_hashes':[],
            'random_weight_consumption':[],'G_anchor_consumption':[],
            'accuracy_computed':False,'validation_arrays_loaded':False,'test_arrays_loaded':False,
            'full_training_allowed':False,'exact_dropout_resume_tested':False,
            'historical_B_bitwise_replay':False,
            'dropout_policy':'C and D rate0.5, stateless route keys; actual masks recorded; new matched runner'}
    try:
        require(CONFIG['arm'] in ('original_gan','matched_ros','random_local','learned_local'),'Unknown arm')
        require(CONFIG['data_mode']=='auto_real_B' and CONFIG['seed']==42,'This integration release locks real B data and seed42')
        unsigned={key:value for key,value in CONFIG.items() if key!='config_sha256'}
        require(json_sha(unsigned)==CONFIG['config_sha256'],'Config integrity failed')
        require(file_sha(__file__)==CONFIG['worker_sha256'],'Worker source integrity failed')
        phase('worker start / NumPy')
        import numpy as numpy
        np=numpy
        run_arm()
    except BaseException as error:
        REPORT.update(status='TIMEOUT' if isinstance(error,TimeoutError) else 'FAILED',**{'pass':False},
                      error_type=type(error).__name__,error=str(error),traceback=traceback.format_exc(),
                      elapsed_seconds=round(time.monotonic()-START,3))
        atomic_json(FOLDER/'arm_report.json',REPORT)
        traceback.print_exc();sys.exit(1)


if __name__=='__main__':
    main()
'''


In [ ]:
# Kaggle-only Mốc 2 launcher. TensorFlow is imported only inside fresh workers.
import datetime as _m2_dt
import hashlib as _m2_hashlib
import json as _m2_json
import os as _m2_os
from pathlib import Path as _M2Path
import shutil as _m2_shutil
import signal as _m2_signal
import subprocess as _m2_subprocess
import sys as _m2_sys
import time as _m2_time
import traceback as _m2_traceback
import uuid as _m2_uuid
import zipfile as _m2_zipfile


_M2_ARMS = ('original_gan', 'matched_ros', 'random_local', 'learned_local')
_M2_GAN_ARMS = ('original_gan', 'learned_local')
_M2_ANCHOR_ARMS = ('matched_ros', 'random_local', 'learned_local')
_M2_LOCAL_ARMS = ('random_local', 'learned_local')
_M2_GATE_KEYS = ('freeze', 'trace_consumption', 'dropout_C_mask_keys',
                 'iteration_skips', 'checkpoint_roundtrip')


def _m2_json_sha(value):
    payload = _m2_json.dumps(value, sort_keys=True, separators=(',', ':'),
                            ensure_ascii=True, allow_nan=False).encode('utf-8')
    return _m2_hashlib.sha256(payload).hexdigest()


def _m2_require(condition, message):
    if not condition:
        raise RuntimeError(message)


def _m2_is_sha(value):
    return (isinstance(value, str) and len(value) == 64
            and all(c in '0123456789abcdef' for c in value))


def _m2_read_json(path):
    value = _m2_json.loads(_M2Path(path).read_text(encoding='utf-8'))
    _m2_require(isinstance(value, dict), f'JSON không phải object: {path}')
    return value


def _m2_atomic_json(path, value):
    path = _M2Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(_m2_json.dumps(value, ensure_ascii=False, indent=2,
                                      allow_nan=False) + '\n', encoding='utf-8')
    temporary.replace(path)


def _m2_verify_arm(folder, config):
    """Validate the current arm's report and actual consumed-record hashes."""
    report_path = folder / 'arm_report.json'
    _m2_require(report_path.is_file(), f'Thiếu arm_report.json: {folder.name}')
    report = _m2_read_json(report_path)
    for key in ('attempt_id', 'arm', 'config_sha256', 'worker_sha256'):
        _m2_require(report.get(key) == config[key], f'Report khác {key}: {folder.name}')
    if report.get('status') == 'TIMEOUT':
        raise TimeoutError(report.get('error', f'Worker {folder.name} hết thời gian'))
    _m2_require(report.get('status') == 'PASS' and report.get('pass') is True,
                f"Worker {folder.name} chưa PASS: {report.get('error', report.get('phase'))}")
    _m2_require(report.get('input', {}).get('real_B_cache_verified') is True,
                f'Worker {folder.name} chưa xác thực cache B gốc')
    for flag in ('accuracy_computed', 'validation_arrays_loaded',
                 'test_arrays_loaded', 'full_training_allowed'):
        _m2_require(report.get(flag) is False, f'Cờ phạm vi thiếu/sai: {folder.name}/{flag}')
    clocks = report.get('clocks', {})
    expected_calls = 68 if config['arm'] in _M2_GAN_ARMS else 4
    _m2_require(clocks.get('C_updates') == 4 and clocks.get('final_clock') == 68
                and clocks.get('real_optimizer_calls') == expected_calls,
                f'Clock/cập nhật không đúng Mốc 2: {folder.name}')
    gates = report.get('gates', {})
    for key in _M2_GATE_KEYS:
        _m2_require(gates.get(key) is True, f'Gate chưa PASS: {folder.name}/{key}')
    initialization = report.get('initialization', {})
    init_keys = ['C_trainable_sha256', 'C_nontrainable_sha256', 'C_adam_slots_sha256']
    if config['arm'] in _M2_GAN_ARMS:
        init_keys.extend(('D_trainable_sha256', 'G_trainable_sha256',
                          'head_dense_trainable_sha256'))
    for key in init_keys:
        _m2_require(_m2_is_sha(initialization.get(key)),
                    f'Thiếu hash khởi tạo: {folder.name}/{key}')

    consumed_path = folder / 'consumed_trace.json'
    _m2_require(consumed_path.is_file(), f'Thiếu consumed_trace.json: {folder.name}')
    consumed = _m2_read_json(consumed_path)
    _m2_require(consumed.get('attempt_id') == config['attempt_id']
                and consumed.get('arm') == config['arm'],
                f'Consumed trace khác attempt/arm: {folder.name}')
    trace = report.get('trace', {})
    _m2_require(trace.get('consumed_loops') == 2,
                f'Worker không tiêu thụ đúng hai loop: {folder.name}')
    for records_key, hash_key, applicable in (
        ('common_c', 'common_C', True),
        ('anchors', 'anchors_C', config['arm'] in _M2_ANCHOR_ARMS),
        ('local_geometry', 'local_geometry', config['arm'] in _M2_LOCAL_ARMS),
        ('common_gan', 'GAN_private', config['arm'] in _M2_GAN_ARMS),
    ):
        records = consumed.get(records_key)
        if applicable:
            _m2_require(isinstance(records, list) and len(records) > 0,
                        f'Thiếu bản ghi đã tiêu thụ: {folder.name}/{records_key}')
            if records_key == 'common_c':
                _m2_require(len(records) == 4,
                            f'Phải có bốn bản ghi C_real/C_fake: {folder.name}')
            if records_key == 'local_geometry':
                _m2_require(len(records) == 2,
                            f'Phải có hai bản ghi local geometry: {folder.name}')
            _m2_require(_m2_is_sha(trace.get(hash_key))
                        and trace[hash_key] == _m2_json_sha(records),
                        f'Hash không khớp actual consumed records: {folder.name}/{records_key}')
        else:
            _m2_require(records in (None, []) and trace.get(hash_key) is None,
                        f'Trace của nhánh không áp dụng phải rỗng: {folder.name}/{records_key}')
    return report


def _m2_verify_pairs(reports):
    _m2_require(set(reports) == set(_M2_ARMS), 'Chưa đủ bốn report Mốc 2')
    verified = {}

    def same(section, key, arms, label):
        values = [reports[arm][section][key] for arm in arms]
        _m2_require(all(value == values[0] for value in values),
                    f'Đối chứng không ghép cặp: {label}')
        verified[label] = True

    for key in ('C_trainable_sha256', 'C_nontrainable_sha256', 'C_adam_slots_sha256'):
        same('initialization', key, _M2_ARMS, 'all_arms_' + key)
    for key in ('D_trainable_sha256', 'G_trainable_sha256', 'head_dense_trainable_sha256'):
        same('initialization', key, _M2_GAN_ARMS, 'GAN_arms_' + key)
    same('trace', 'common_C', _M2_ARMS, 'actual_common_C_all_arms')
    same('trace', 'anchors_C', _M2_ANCHOR_ARMS, 'actual_anchors_ROS_random_learned')
    same('trace', 'local_geometry', _M2_LOCAL_ARMS, 'actual_local_geometry_random_learned')
    same('trace', 'GAN_private', _M2_GAN_ARMS, 'actual_GAN_private_original_learned')
    return verified


def _m2_verify_available_pairs(reports):
    """Fail at the first available mismatch, before spending the next arm."""
    checked = {}
    groups = (
        ('initialization', ('C_trainable_sha256','C_nontrainable_sha256','C_adam_slots_sha256'), _M2_ARMS),
        ('initialization', ('D_trainable_sha256','G_trainable_sha256','head_dense_trainable_sha256'), _M2_GAN_ARMS),
        ('trace', ('common_C',), _M2_ARMS),
        ('trace', ('anchors_C',), _M2_ANCHOR_ARMS),
        ('trace', ('local_geometry',), _M2_LOCAL_ARMS),
        ('trace', ('GAN_private',), _M2_GAN_ARMS),
    )
    for section, keys, arms in groups:
        available = [arm for arm in arms if arm in reports]
        if len(available) < 2:
            continue
        for key in keys:
            values = [reports[arm][section][key] for arm in available]
            _m2_require(all(value == values[0] for value in values),
                        f'Ghép cặp lệch ngay sau {available[-1]}: {section}/{key}; không chạy thêm nhánh.')
            checked[section+'/'+key] = available
    return checked


def launch_moc2():
    # One absolute cutoff covers ALL four workers, hashes, TF imports and compile.
    started = _m2_time.monotonic()
    if not _m2_sys.platform.startswith('linux') or not _M2Path('/kaggle/working').is_dir():
        raise RuntimeError('Mốc 2 chỉ chạy trên Kaggle; không chạy ở máy cá nhân.')
    if type(TIME_LIMIT_SECONDS) is not int or not 1 <= TIME_LIMIT_SECONDS <= 1800:
        raise ValueError('TIME_LIMIT_SECONDS phải trong 1..1800; tổng bốn nhánh tối đa 30 phút.')
    if type(GPU_INDEX) is not int or GPU_INDEX < 0:
        raise ValueError('GPU_INDEX phải là số nguyên không âm')
    if type(SEED) is not int or SEED != 42:
        raise ValueError('Mốc 2 khóa SEED=42; không tìm seed hoặc tự chạy lại')
    if B_RUN is not None and not isinstance(B_RUN, str):
        raise ValueError('B_RUN phải là None hoặc đường dẫn dạng chuỗi')
    if not isinstance(WORKER_SOURCE, str) or not WORKER_SOURCE.strip():
        raise ValueError('Thiếu WORKER_SOURCE đã đóng gói trong notebook')
    deadline = started + TIME_LIMIT_SECONDS
    stamp = _m2_dt.datetime.now(_m2_dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    attempt_id = stamp + '_' + _m2_uuid.uuid4().hex[:8]
    folder = _M2Path('/kaggle/working') / ('MOC2_IP_SMOKE_' + attempt_id)
    folder.mkdir(exist_ok=False)
    worker_path = folder / 'worker.py'
    worker_path.write_text(WORKER_SOURCE, encoding='utf-8')
    worker_hash = _m2_hashlib.sha256(worker_path.read_bytes()).hexdigest()
    result = {
        'schema': 3, 'attempt_id': attempt_id, 'status': 'RUNNING', 'pass': False,
        'worker_sha256': worker_hash, 'scope': 'Moc2 four-arm two-loop integration only',
        'data_mode': 'auto_real_B', 'seed': SEED, 'gpu_index': GPU_INDEX,
        'limit_seconds': TIME_LIMIT_SECONDS, 'arm_order': list(_M2_ARMS),
        'C_updates_per_arm': 4, 'final_Adam_clock_per_arm': 68,
        'accuracy_computed': False, 'validation_arrays_loaded': False,
        'test_opened': False, 'full_training_allowed': False,
        'watchdog': 'independent GNU timeout; one shared absolute deadline',
        'arms': {}, 'paired_gates': {}, 'cleanup_errors': [], 'log_read_errors': [],
    }
    _m2_atomic_json(folder / 'result.json', result)
    process = None
    current_arm = None
    current_folder = None
    offset = 0

    def read_log_tail():
        nonlocal offset
        if current_folder is None:
            return
        log_path = current_folder / 'worker.log'
        try:
            if log_path.is_file():
                with log_path.open('r', encoding='utf-8', errors='replace') as log:
                    log.seek(offset)
                    content = log.read(65536)
                    offset = log.tell()
                if content:
                    print(content, end='', flush=True)
        except OSError as error:
            result['log_read_errors'].append({'arm': current_arm, 'error': str(error)})

    def kill_worker():
        if process is not None and process.poll() is None:
            try:
                _m2_os.killpg(process.pid, _m2_signal.SIGKILL)
            except ProcessLookupError:
                pass
            except OSError as error:
                result['cleanup_errors'].append({'arm': current_arm, 'error': str(error)})
            try:
                process.wait(timeout=5)
            except (_m2_subprocess.TimeoutExpired, OSError) as error:
                result['cleanup_errors'].append({'arm': current_arm, 'error': str(error)})

    def check_deadline():
        if _m2_time.monotonic() >= deadline:
            raise TimeoutError('Đã tới deadline tổng Mốc 2; không khởi động thêm nhánh.')

    reports = {}
    try:
        print('MỐC 2:', folder, flush=True)
        print(f'Bốn nhánh tuần tự trên GPU {GPU_INDEX}; tổng deadline {TIME_LIMIT_SECONDS}s; '
              'mỗi nhánh hai loop, không accuracy/validation/test.', flush=True)
        timeout_executable = _m2_shutil.which('timeout')
        _m2_require(timeout_executable is not None,
                    'Thiếu GNU timeout; không khởi động GPU worker.')
        env = dict(_m2_os.environ)
        env.update(CUDA_VISIBLE_DEVICES=str(GPU_INDEX), PYTHONUNBUFFERED='1',
                   TF_USE_LEGACY_KERAS='1', PYTHONHASHSEED=str(SEED), LC_ALL='C')
        for arm in _M2_ARMS:
            check_deadline()
            current_arm = arm
            current_folder = folder / arm
            current_folder.mkdir(exist_ok=False)
            offset = 0
            process = None
            config = {
                'schema': 3, 'attempt_id': attempt_id, 'arm': arm,
                'attempt_dir': str(current_folder), 'suite_attempt_dir': str(folder),
                'seed': SEED, 'data_mode': 'auto_real_B', 'b_run': B_RUN,
                'gpu_index': GPU_INDEX, 'time_limit_seconds': TIME_LIMIT_SECONDS,
                'start_monotonic': started, 'deadline_monotonic': deadline,
                'worker_sha256': worker_hash,
            }
            config['config_sha256'] = _m2_json_sha(config)
            config_path = current_folder / 'config.json'
            _m2_atomic_json(config_path, config)
            result['arms'][arm] = {'status': 'RUNNING', 'pass': False,
                                   'config_sha256': config['config_sha256']}
            _m2_atomic_json(folder / 'result.json', result)
            print(f'\nNHÁNH {len(reports) + 1}/4: {arm} | '
                  f'còn {max(0., deadline - _m2_time.monotonic()):.1f}s trong deadline chung',
                  flush=True)
            with (current_folder / 'worker.log').open('w', encoding='utf-8') as output:
                # No --foreground: monitor and child share this new process group.
                # The independent monitor also survives a killed notebook kernel.
                remaining = deadline - _m2_time.monotonic() - .5
                if remaining <= 0:
                    raise TimeoutError('Hết thời gian trước khi khởi động watchdog nhánh tiếp theo')
                process = _m2_subprocess.Popen(
                    [timeout_executable, '--verbose', '--signal=KILL', f'{remaining:.6f}s',
                     _m2_sys.executable, '-u', str(worker_path), str(config_path)],
                    stdout=output, stderr=_m2_subprocess.STDOUT, env=env,
                    start_new_session=True)
                last_heartbeat = _m2_time.monotonic()
                while True:
                    check_deadline()
                    if process.poll() is not None:
                        break
                    read_log_tail()
                    now = _m2_time.monotonic()
                    if now - last_heartbeat >= 45:
                        print(f'{arm}: worker còn chạy | tổng {now - started:.0f}/'
                              f'{TIME_LIMIT_SECONDS}s', flush=True)
                        last_heartbeat = now
                    _m2_time.sleep(min(1., max(.01, deadline - _m2_time.monotonic())))
            read_log_tail()
            check_deadline()
            log_text = (current_folder / 'worker.log').read_text(encoding='utf-8', errors='replace')
            if process.returncode in (124, 137, -9) and 'sending signal KILL' in log_text:
                raise TimeoutError(f'Watchdog đã dừng {arm} khi chạm deadline tổng')
            report = _m2_verify_arm(current_folder, config)
            _m2_require(process.returncode == 0,
                        f'Worker {arm} thoát lỗi: exit {process.returncode}; không chấp nhận PASS')
            reports[arm] = report
            result['arms'][arm] = {'status': 'PASS', 'pass': True,
                                   'worker_exit_code': process.returncode,
                                   'config_sha256': config['config_sha256'],
                                   'worker_report': report}
            result['paired_gates_so_far'] = _m2_verify_available_pairs(reports)
            _m2_atomic_json(folder / 'result.json', result)
            print(f'{arm}: PASS kỹ thuật; không phải kết quả accuracy.', flush=True)
        check_deadline()
        result['paired_gates'] = _m2_verify_pairs(reports)
        check_deadline()
        result.update(status='PASS', **{'pass': True})
    except BaseException as error:
        kill_worker()
        status = ('TIMEOUT' if isinstance(error, TimeoutError)
                  else 'ABORTED' if isinstance(error, KeyboardInterrupt) else 'FAILED')
        result.update(status=status, **{'pass': False}, error=str(error),
                      traceback=_m2_traceback.format_exc())
        if current_arm is not None and result['arms'].get(current_arm, {}).get('status') == 'RUNNING':
            result['arms'][current_arm].update(status=status, error=str(error))
            report_path = current_folder / 'arm_report.json'
            if report_path.is_file():
                try:
                    result['arms'][current_arm]['worker_report_unvalidated'] = _m2_read_json(report_path)
                except (OSError, ValueError, RuntimeError):
                    pass
    finally:
        kill_worker()
        read_log_tail()
        if result['cleanup_errors']:
            result.update(status='FAILED', **{'pass': False},
                          error='Không xác minh được cleanup; xem cleanup_errors.')
        result['elapsed_seconds'] = round(_m2_time.monotonic() - started, 3)
        result['last_worker_exit_code'] = None if process is None else process.returncode
        result['completed_arm_count'] = len(reports)
        _m2_atomic_json(folder / 'result.json', result)
        archive_path = folder.with_name(folder.name + '_REPORT.zip')
        temporary_archive = archive_path.with_suffix('.zip.tmp')
        with _m2_zipfile.ZipFile(temporary_archive, 'w', _m2_zipfile.ZIP_DEFLATED) as archive:
            for name in ('result.json', 'worker.py'):
                path = folder / name
                if path.is_file():
                    archive.write(path, folder.name + '/' + name)
            for arm in _M2_ARMS:
                for name in ('config.json', 'worker.log', 'arm_report.json', 'consumed_trace.json', 'schedule_spec.json'):
                    path = folder / arm / name
                    if path.is_file():
                        archive.write(path, folder.name + '/' + arm + '/' + name)
        temporary_archive.replace(archive_path)
        print('\nKẾT THÚC MỐC 2:', result['status'], '|', result['elapsed_seconds'], 's', flush=True)
        print('Báo cáo chính:', folder / 'result.json', flush=True)
        print('ZIP NHỎ:', archive_path, flush=True)
        try:
            from IPython.display import display, FileLink
            display(FileLink(str(archive_path)))
        except ImportError:
            pass
    if result['pass'] is not True:
        raise RuntimeError(f"Mốc 2 {result['status']}; đã lưu báo cáo, không tự chạy lại. "
                           f"{result.get('error', '')}")
    return result


moc2_result = launch_moc2()


## Đọc kết quả và dừng đúng mốc

`result.json` ở thư mục suite là kết quả cuối. PASS yêu cầu đủ bốn arm và các paired gates, không chỉ bốn dòng “ARM PASS”. ZIP chứa config, source, báo cáo từng nhánh và consumed traces; không chứa weights lớn.

PASS chỉ xác nhận runner thực thi đúng các yếu tố ghép cặp đã kiểm trong hai loop này. Nó **chưa đo** Macro-F1, lợi ích mẫu sinh, convergence, hoặc tính mới. Chưa kiểm resumed trajectory. `full_training_allowed=false` vẫn giữ nguyên.

Sau khi đọc ZIP và chốt runner, mới benchmark chi phí ổn định và lập ngân sách pilot. Không tự mở campaign nhiều seed từ notebook này.
